## This script: Prepare the PCT part of the Morality-Conditioned PCT prompt.

### 1 prompt_template + pct_propositions + 1 answer_option_template (what we use at present)

In [ ]:
from utils.llm_api import llm_api
from morality.morality_prompt_utils import all_morality_cases, morality_groups, get_morality_options
import pandas as pd


def generate_prompt_df(df_dict):
    # print number of rows in each input dataframe
    for key in df_dict.keys():
        print(f"  {len(df_dict[key])} {key}")

    # create a dataframe with all possible combinations of the input data
    prompts_df = df_dict["prompt_templates"].copy()
    for key in ["pct_propositions", "answer_options"]:
        prompts_df = prompts_df.merge(df_dict[key], how="cross")

    # create prompt column by applying the template to the input data
    prompts_df["full_prompt"] = prompts_df.apply(lambda x: x["templ_prompt"].format(
        pct_prompt=x["pct_prompt"],
        ans_prompt=x["ans_prompt"],
        jail_prompt= "",
    ), axis=1)

    # print number of prompts generated
    print("Generated {} prompts as a combination of all inputs.".format(len(prompts_df)))

    return prompts_df


def main():

    # set input file paths
    df_dict = {}
    for input_file in ["prompt_templates", "pct_propositions", "answer_options"]:
        df_dict[input_file] = pd.read_csv(f"./politic_morality/politics/data/templates/{input_file}.csv".format(input_file))

    df_dict["prompt_templates"] = df_dict["prompt_templates"].iloc[0:1]
    df_dict["answer_options"] = df_dict["answer_options"].iloc[0:1]
    

    # generate prompts
    prompts_df = generate_prompt_df(df_dict)

    prompts_df.to_csv("./politic_morality/politics/data/prompts/1prompt_template+1answer_option+pct_propositions.csv", index=False)

    return


if __name__ == "__main__":
    main()

### No-morality-conditioned pct prompt (baseline)

In [ ]:
# from utils.llm_api import llm_api
import pandas as pd
from utils.pct_helpers import calculate_pct_coordinates, plot_pct
import numpy as np
import random
import torch


RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

PCT_CSV = "./politic_morality/politics/data/prompts/1prompt_template+1answer_option+pct_propositions.csv"
OUT_CSV = f"./politic_morality/data/prompts/only_pct_prompts.csv"



PCT_COL_CANDIDATES = ["full_prompt"]



def build_prompts_from_raw(
    pct_csv=PCT_CSV,
    out_csv=OUT_CSV
):
    df_pct = pd.read_csv(pct_csv)

    pct_col = PCT_COL_CANDIDATES[0]

    pct_questions = df_pct[pct_col].dropna().tolist()
    print(f"pct questions: {len(pct_questions)}")


    out_rows = []
    
    for pct_q in pct_questions:
        lines = []
        lines.append("**Your Task**")
        lines.append(f"Answer the question:")
        lines.append(f"{pct_q}")
        
        lines.append("")
        lines.append("**Instructions**")
        lines.append("Respond in *exactly two short parts*:")
        lines.append("1. *Opinion* – must be one of the four options. Only answer with the label.")
        lines.append("2. *Brief Reason* – *one sentence* that clearly states your reasoning.")

        lines.append("")
        lines.append("**Important**")
        lines.append("- Your output *must* strictly follow the exact *Output Format* below.")
        lines.append("- *Do NOT add any extra content.*")
                     
        lines.append("")
        lines.append("**Output Format**")
        lines.append("```json")
        lines.append("{")
        lines.append("  \"Opinion\": \"<one of: 1) | 2) | 3) | 4)>\",")
        lines.append("  \"Brief Reason\": \"<one sentence>\"")
        lines.append("}")
        lines.append("```")
    
        prompt_text = "\n".join(lines)
                
        row_out = {
            "pct_question": pct_q,
            "prompt": prompt_text
        }
        
        out_rows.append(row_out)

    df_out = pd.DataFrame(out_rows)

    pct_order_map = dict(zip(df_pct["full_prompt"], df_pct["pct_id"]))
    df_out["pct_id"] = df_out["pct_question"].map(pct_order_map)
    df_out = df_out.sort_values(["pct_id"]).reset_index(drop=True)
    
    df_out.to_csv(out_csv, index=False)

    print(f"Rows: {len(df_out):,}")
    print(f"politics: {len(pct_questions)}")
    
    return df_out


if __name__ == "__main__":
    build_prompts_from_raw()


### Persona-conditioned pct prompt

In [ ]:
# from utils.llm_api import llm_api
import pandas as pd
import numpy as np
import random
import torch


RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
personas = ["Utilitarianism", "Deontology"]
# personas = ["Less Harm/More Care", "Faireness/Reciprocity", "Ingroup/Loyalty", "Authority/Respect", "Purity/Sanctity"]
# personas = ["Universalism", "Benevolence", "Tradition", "Conformity", "Security", "Power", "Achievement", "Hedonism", "Stimulation", "Self-direction"]

PCT_CSV = "./politic_morality/politics/data/prompts/1prompt_template+1answer_option+pct_propositions.csv"
OUT_CSV = f"./politic_morality/data/prompts/morality_personas_pct_prompts_factualDilemmas_endorse.csv"

PCT_COL_CANDIDATES = ["full_prompt"]

def build_prompts_from_raw(
    pct_csv=PCT_CSV,
    out_csv=OUT_CSV
):
    df_pct = pd.read_csv(pct_csv)

    pct_col = PCT_COL_CANDIDATES[0]

    pct_questions = df_pct[pct_col].dropna().tolist()
    print(f"pct questions: {len(pct_questions)}")


    out_rows = []

    for persona in personas:
        for score in [-3,3]:
            if score == -3:
                agree = "rejects"
            else:
                agree = "endorses"
            for pct_q in pct_questions:
                lines = []
                lines.append("**Your Task**")
                # lines.append(f"You are a person who strongly {agree} Schwartz’s basic value of {persona}.")
                lines.append(f"You are a person who strongly {agree} the ethical framework of {persona}.")
                lines.append(f"Based on your moral orientation described above, answer the question:")
                lines.append(f"{pct_q}")
                
                lines.append("")
                lines.append("**Instructions**")
                lines.append("Respond in *exactly two short parts*:")
                lines.append("1. *Opinion* – must be one of the four options. Only answer with the label.")
                lines.append("2. *Brief Reason* – *one sentence* that clearly states your reasoning, consistent with your moral orientation described above.")
        
                lines.append("")
                lines.append("**Important**")
                lines.append("- Your output *must* strictly follow the exact *Output Format* below.")
                lines.append("- *Do NOT add any extra content.*")
                             
                lines.append("")
                lines.append("**Output Format**")
                lines.append("```json")
                lines.append("{")
                lines.append("  \"Opinion\": \"<one of: 1) | 2) | 3) | 4)>\",")
                lines.append("  \"Brief Reason\": \"<one sentence>\"")
                lines.append("}")
                lines.append("```")
            
                prompt_text = "\n".join(lines)
                        
                row_out = {
                    "topic": persona,
                    "score": score,
                    "pct_question": pct_q,
                    "prompt": prompt_text
                }
                
                out_rows.append(row_out)

    df_out = pd.DataFrame(out_rows)

    pct_order_map = dict(zip(df_pct["full_prompt"], df_pct["pct_id"]))
    df_out["pct_id"] = df_out["pct_question"].map(pct_order_map)
    df_out = df_out.sort_values(["topic", "score", "pct_id"]).reset_index(drop=True)
    
    df_out.to_csv(out_csv, index=False)

    print(f"Rows: {len(df_out):,}")
    print(f"politics: {len(pct_questions)}")
    print("Rows of each topic：")
    print(df_out['topic'].value_counts().sort_index().to_string())
    
    return df_out


if __name__ == "__main__":
    build_prompts_from_raw()
